# Epsilon Fund — XS Momentum (Rolling Sharpe + Inverse-Vol + Continuous Regime Tilt)

`xs_1` family with the same composable layers as `xs_3_i_r`, but with a
**different ranking signal**:

- **xs_3 family** (residual-Sharpe): strips BTC β before computing Sharpe.
- **xs_1 family** (rolling-Sharpe, this notebook): Sharpe of *raw* returns
  over the J-bar window. Vol-normalised momentum without β stripping.

**Layer stack:**
1. Rolling Sharpe of raw returns (`signal_kind='rolling_sharpe'` in the factory)
2. Inverse-vol weighting within each leg (`_i`)
3. Continuous regime tilt (`_r`) — EMA-19 sets direction, ADX-14 sets intensity
   (linear ramp, saturating at ADX = 50, asymmetric MAX)

**Tilt mechanics (same as xs_3_i_r — continuous intensity):**

| BTC direction + ADX | long_weight | short_weight | Net tilt |
|---|---|---|---|
| Bull, ADX = 50+ (saturated) | 0.55  | 0.45  | +0.05 |
| Bull, ADX ≈ 25 (median)     | 0.525 | 0.475 | +0.025 |
| Bear, ADX ≈ 25 (median)     | 0.475 | 0.525 | −0.05 |
| Bear, ADX = 50+ (saturated) | 0.40  | 0.60  | −0.10 |

**Comparison value:** direct A/B test against `xs_3_i_r` of whether stripping
BTC β adds anything in this universe / regime mix. If results are similar,
β stripping is redundant and you should pick the simpler notebook (xs_1
family) for production.

**`BURNIN_BARS = 50`** (single-rolling warmup = `J + 1` ≤ 46 < 50). Same as
xs_1 / xs_2 / xs_momentum_wf — apples-to-apples comparison vs the simple
xs_1 baseline. Different from xs_3 family's 95, so direct grid-aligned
comparison vs xs_3_i_r will have small grid-shift noise (5-15% Sharpe);
the signal-difference effect should dominate that.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = os.path.join('C:\\', 'Users', 'user', 'Documents', 'Epsilon Fund', 'Epsilon-Quant-Research')
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'backtester'))
sys.path.insert(0, os.path.join(ROOT, 'topics', 'momentum', 'xs_momentum'))
sys.path.insert(0, os.path.join(ROOT, 'topics', 'momentum', 'xs_momentum', 'universe'))

from binance_client  import get_binance_client, get_data
from wf_engine       import (walk_forward, plateau_analysis, plateau_summary,
                              perturbation_test, cost_stress_test)
from wf_visualizer   import plot_walk_forward_results, plot_plateau_analysis
from ls_diagnostics  import (compute_attribution, plot_attribution,
                              bear_hedge_diagnostic, regime_quadrant_diagnostic)
from xs_strategy     import (compute_btc_regime_tilt_panel,
                              summarize_tilt_distribution, make_xs_strategy)
from engine          import backtest
from xs_data         import load_xs_data, build_close_panel, DEFAULT_COINS
from universe_filter import load_cache, get_universe, precompute_avg_volume


---
## Data

Two modes, controlled by `USE_DYNAMIC_UNIVERSE` in the next cell:

**Static mode (`False`)** — fast iteration with a hardcoded coin list (`DEFAULT_COINS`).
Pulls daily OHLCV directly from Binance via `load_xs_data()`. Good for quick parameter
sweeps where the universe is fixed.

**Dynamic mode (`True`)** — universe is re-selected at each rebalance from a local
parquet cache built by `universe_cache.py`. The cache holds **USDT perpetual futures
data only** — perp prices feed both the ranking signal and the backtest PnL (matching
what would actually trade on Hyperliquid for the dollar-neutral L/S strategy). At each
rebalance, `get_universe(dates[r-1], ...)` filters by 30-day avg perp volume and listing
age — strictly no lookahead since only data up to the signal formation date is used.

**Cache maintenance:** run `python universe_cache.py` once to build the cache, then
once per day to incrementally append yesterday's candle. Notebook runs make zero
API calls — they read parquet directly.

**`COST`** — per-leg trading cost fraction. The strategy returns gross
`strategy_returns` plus a `turnover` column; the engine applies
`trade_cost = turnover × COST` automatically. Adjust `COST` without re-running
the strategy — `cost_stress_test()` rescales it for free.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODE
# ══════════════════════════════════════════════════════════════════════════════
USE_DYNAMIC_UNIVERSE = True   # True  → load from perp cache (universe_cache.py)
                               # False → fetch API with COINS list below

# ══════════════════════════════════════════════════════════════════════════════
# UNIVERSE FILTER CONFIG  (used when USE_DYNAMIC_UNIVERSE = True)
# ══════════════════════════════════════════════════════════════════════════════
UF_TOP_N          = 30          # max coins in the eligible universe at each rebalance
UF_MIN_VOLUME     = 80_000_000  # min 30-day avg daily USDT perp volume ($)
UF_MIN_AGE_DAYS   = 90         # min days since listing (perp onboard date)
UF_VOLUME_WINDOW  = 30          # rolling window for avg volume computation

# ══════════════════════════════════════════════════════════════════════════════
# STATIC CONFIG  (used when USE_DYNAMIC_UNIVERSE = False)
# ══════════════════════════════════════════════════════════════════════════════
COINS    = DEFAULT_COINS   # ['BTC','ETH','SOL','BNB','XRP','DOGE','ADA','AVAX','LINK','MATIC']
INTERVAL = '1d'
LOOKBACK = 2200            # days; covers ~6 years for most coins

# ══════════════════════════════════════════════════════════════════════════════
# SHARED
# ══════════════════════════════════════════════════════════════════════════════
COST = 0.001   # per-leg trading cost fraction



# ── Inverse-vol weighting (intra-leg risk parity) ────────────────────────
# Within each long / short basket, weight each coin by 1/σ_coin (normalised
# so leg weights sum to 1).  σ_coin = rolling stdev of daily returns over
# IV_VOL_WINDOW bars.  Reduces single-coin tail risk (memecoin pumps) and
# brings risk contributions toward parity within each leg.  Literature
# (Moreira-Muir 2017, Asness/Frazzini/Pedersen): adds 0.1-0.3 Sharpe and
# cuts max drawdown ~20-40% on momentum baskets vs equal-weight.
IV_VOL_WINDOW = 30   # days; 30 matches the universe-filter volume window

# ── Regime-tilted asymmetric leg sizing (Layer 1: BTC direction → leg balance) ─
# Bear regime → tilt toward shorts (active hedge value).
# Bull regime → mild tilt toward longs (don't fight bull, but don't lever up).
# Neutral     → 50/50 dollar-neutral baseline (no conviction → no tilt).
# ASYMMETRIC magnitude (bear > bull) reflects this strategy's role as a HEDGE
# to long-only momentum books — defense first, participation second.
# These are STRUCTURAL design choices, NOT Optuna-tunable hyperparameters.
MAX_BEAR_TILT = 0.2   # bear: long_weight = 0.40, short_weight = 0.60
MAX_BULL_TILT = 0.2   # bull: long_weight = 0.55, short_weight = 0.45

# BTC regime classifier — CONTINUOUS-INTENSITY TILT
# Direction comes from EMA position (binary: bull vs bear, no neutral state).
# Intensity comes from ADX magnitude, scaled linearly to [0, 1] saturating at
# REGIME_ADX_SCALE.  Replaces the earlier binary threshold setup — no magic
# trend-strength cutoff, smooth tilt magnitude transitions.
REGIME_EMA_PERIOD = 19
REGIME_ADX_PERIOD = 14
REGIME_ADX_SCALE  = 40    # ADX value at which tilt saturates at MAX_*_TILT.
                          # Below: linear ramp.  Above: capped at MAX.
                          # (E.g. ADX=25 → 50% of MAX; ADX=50+ → full MAX.)

# ── Static data load (API, only when USE_DYNAMIC_UNIVERSE = False) ────────────
if not USE_DYNAMIC_UNIVERSE:
    client = get_binance_client()
    print('Fetching daily OHLCV...')
    data = load_xs_data(coins=COINS, interval=INTERVAL, lookback=LOOKBACK, client=client)

    panel    = build_close_panel(data)
    volume   = None    # signals no dynamic filtering inside the strategy
    meta     = None

    ew_equity = (1 + panel.pct_change().mean(axis=1).fillna(0)).cumprod()
    ew_df     = pd.DataFrame({'Close': ew_equity}, index=panel.index)
    driver_df = data['BTC'][['Close']].copy()

    print(f'\nClose panel: {panel.shape}  ({panel.index[0].date()} → {panel.index[-1].date()})')
    print(f'Driver (BTC): {len(driver_df)} bars')

In [ ]:
if USE_DYNAMIC_UNIVERSE:
    # ── Load from parquet cache (no API calls) ────────────────────────────────
    # Run  python universe_cache.py  first to build / update the cache.
    print('Dynamic universe mode — loading perp cache...')
    close_full, volume, meta = load_cache()

    print(f'  close  : {close_full.shape}')
    print(f'  volume : {volume.shape}')
    print(f'  meta   : {meta.shape}')

    # ── Clip price panel to LOOKBACK days from the most recent cached date ───
    # `volume` and `meta` intentionally KEEP full history so get_universe's
    # rolling 30-day avg has enough bars even when as_of_date sits at the
    # start of the LOOKBACK window.  Slicing by `as_of_date <= dates[r-1]`
    # inside get_universe still prevents lookahead.
    cutoff     = close_full.index[-1] - pd.Timedelta(days=LOOKBACK)
    close_full = close_full.loc[close_full.index >= cutoff]

    # ── Restrict panel to coins with a recorded listing date in meta ─────────
    tradeable = sorted([c for c in close_full.columns
                         if c in meta.index and pd.notna(meta.loc[c, 'listing_date'])])
    panel = close_full[tradeable]

    # ── Equal-weight basket benchmark ────────────────────────────────────────
    ew_equity = (1 + panel.pct_change().mean(axis=1).fillna(0)).cumprod()
    ew_df     = pd.DataFrame({'Close': ew_equity}, index=panel.index)

    # ── Driver (BTC perp — provides the walk-forward date index) ─────────────
    driver_df = panel[['BTC']].rename(columns={'BTC': 'Close'}).copy()

    print(f'\nTradeable universe: {len(tradeable)} coins')
    print(f'Date range: {panel.index[0].date()} → {panel.index[-1].date()}  '
          f'({len(panel)} bars, clipped to LOOKBACK={LOOKBACK})')
    print(f'volume kept full ({len(volume)} bars) for clean rolling-avg lookback')
else:
    print('Static mode — data already loaded in Cell 3.')

---
## Regime panel computation (BTC OHLC → continuous-intensity tilt)

Fetches BTC's full OHLC from Binance once at notebook setup (sub-second). Computes:
- **EMA-19** of BTC close → **direction** of tilt (above EMA = bull, below = bear)
- **ADX-14** (Wilder's smoothing) → **intensity** of tilt (linear ramp,
  saturating at `REGIME_ADX_SCALE = 50`)

**Continuous-intensity formulation** (replaces the earlier binary threshold setup):

```
direction = +1 if BTC > EMA-19 else −1     (binary)
intensity = min(ADX / SCALE, 1.0)           (continuous, ∈ [0, 1])
tilt      = direction × intensity × MAX_*_TILT   (asymmetric per direction)
```

**Why continuous instead of threshold-binary:**
- No magic number for "trend strong enough to act on" — ADX magnitude maps
  smoothly to tilt magnitude. ADX = 35 → 70% tilt; ADX = 50+ → full tilt.
- Always in either bull or bear regime (`regime_tilt_panel` is non-zero on
  effectively every bar). The only zero-tilt boundary is the EMA crossing
  itself, and ADX is essentially never literally 0 in practice.
- Smoother weight transitions → no abrupt portfolio re-tilting when ADX
  crosses a threshold → less spurious rebalance turnover.

**ADX scaling (`REGIME_ADX_SCALE = 50`):**
- `ADX = 50` saturates at MAX (full tilt). Empirically a strong-trend value.
- `ADX = 25` (typical median) → 50% of MAX tilt.
- `ADX = 10` (chop) → 20% of MAX tilt — small but non-zero.
- `ADX > 50` capped at 1.0 — no extra reward for extreme ADX.

The asymmetric `MAX_BEAR_TILT = 0.10` vs `MAX_BULL_TILT = 0.05` continues to
apply at the cap. So tilt magnitudes range:
- Bull direction: 0 → +0.05 (0.50/0.50 → 0.55/0.45)
- Bear direction: 0 → −0.10 (0.50/0.50 → 0.40/0.60)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BTC regime panel — fetched OHLC + infra continuous-intensity tilt helper
# ══════════════════════════════════════════════════════════════════════════════
if 'client' not in globals():
    client = get_binance_client()

print('Fetching BTC OHLC for regime classifier...')
btc_ohlc = get_data(client, 'BTCUSDT', '1d', LOOKBACK)
print(f'  BTC OHLC: {btc_ohlc.shape}  ({btc_ohlc.index[0].date()} → {btc_ohlc.index[-1].date()})')

regime_tilt_panel = compute_btc_regime_tilt_panel(
    btc_ohlc,
    ema_period    = REGIME_EMA_PERIOD,
    adx_period    = REGIME_ADX_PERIOD,
    adx_scale     = REGIME_ADX_SCALE,
    max_bull_tilt = MAX_BULL_TILT,
    max_bear_tilt = MAX_BEAR_TILT,
)

# Diagnostic distribution + ADX summary
from xs_strategy import wilder_adx
btc_adx = wilder_adx(btc_ohlc['High'], btc_ohlc['Low'], btc_ohlc['Close'],
                     period=REGIME_ADX_PERIOD)
summarize_tilt_distribution(regime_tilt_panel,
                            max_bull_tilt=MAX_BULL_TILT,
                            max_bear_tilt=MAX_BEAR_TILT,
                            btc_adx=btc_adx)

# `regime_label` retained for any cells that consume it as a sign-only marker
regime_label = regime_tilt_panel.apply(
    lambda t: 'bull' if t > 0 else ('bear' if t < 0 else 'neutral')
)


---
## ⚠️  WF Engine Interface — Multi-Asset Adapter

### What the engine expects

`walk_forward()` was designed for single-asset strategies:
```
strategy_fn(df_slice, params)  →  (df, indicator_cols)
```
where `df_slice` is **one coin's** OHLCV for the training or test window, and the
returned `df` must carry a `position` column (`1`/`0`/`-1`). The engine slices a single
`df` by bar index to build train/test windows.

### Why XS momentum doesn't fit

1. **Multi-asset data** — `strategy_fn` receives only one coin's slice, but XS
   momentum needs all coins simultaneously for cross-sectional ranking.
2. **Portfolio returns, not a single position** — the output is a weighted long/short
   portfolio return; there is no single `1`/`0`/`-1` column that makes sense.

### The adapter used here (no engine modifications)

**Closure + `strategy_returns` path:**

- `make_xs_strategy(full_panel, cost_per_leg)` captures the full multi-asset close
  panel as a Python closure.
- Inside `strategy_fn`, `df_slice` is ignored **except for its `DatetimeIndex`**, which
  is used to window `full_panel` to the correct train/test dates.
- The strategy computes per-bar portfolio returns (including transaction costs) and
  stores them in a `strategy_returns` column.
- `position = 1` is set everywhere, activating the engine's precomputed-returns path:
  ```python
  # engine.py — precomputed path
  net_returns  = position.shift(1) * strategy_returns   # bar-0 zeroed (see below)
  equity_curve = (1 + net_returns).cumprod()
  ```
- `cost=0` is passed to `walk_forward` because costs are **already baked** into
  `strategy_returns`. Passing any non-zero cost would double-count.

### Three consequences to be aware of

**1. Bar-0 of every slice has zero return.**  
The engine applies `eff_pos = position.shift(1)`, so `eff_pos[0] = 0` regardless of
`position[0] = 1`. This zeros bar-0's return in every train and test window. On a
daily strategy with 700+ bars per fold the effect is negligible (~0.14% of bars).

**2. `num_trades = 0` — custom `reject_fn` is required.**  
The engine detects trades via `position.diff()`. With `position=1` always,
`position.diff() = 0` everywhere, so no trades are logged. The **default** `reject_fn`
has `if num_trades < 7: return True` which would reject every Optuna trial. The
custom `reject_fn` below uses Sharpe and drawdown filters instead.

**3. `cost_stress_test()` is replaced by a manual version.**  
The built-in `cost_stress_test()` re-runs the OOS DataFrame at higher costs by
re-calling `engine.backtest()`, but because `position=1` always the engine sees no
position changes and adds zero cost. The manual stress test re-runs `strategy_fn`
at escalating `cost_per_leg` values on the OOS date window instead.

### What a proper multi-asset engine would need

If the XS strategy family grows, the engine would need these targeted changes:

| Change | File | Where |
|--------|------|-------|
| Accept `data_dict` alongside `df` so folds slice all coins | `wf_engine.py` | `walk_forward()` signature |
| Pass the full per-coin slice dict to `strategy_fn` | `wf_engine.py` | `_make_objective()` and fold loop |
| Allow `strategy_fn` to return a position *matrix* (bars × coins) | `wf_engine.py` | return contract |
| Aggregate portfolio returns from the position matrix before calling `backtest()` | `wf_engine.py` | `_run_backtest()` |

The closure adapter keeps the engine untouched and is fully sufficient for a
single XS strategy. Consider the engine changes only if you need the WF framework
to also optimise per-coin position sizing or dynamic universe selection.

---

---
## Parameter Configuration

Define which parameters to optimise and which to anchor.

**First run:** leave `FIXED_PARAMS` empty — let Optuna search all three dimensions
freely. After the stability table prints, copy parameters with CV < 0.15 into
`FIXED_PARAMS` and re-run.

| Param | Range | Notes |
|---|---|---|
| `J` | 7–30 days | Standard daily momentum formation window (1 week to 1 month) |
| `K` | 7–14 days | Holding period; shorter = more rebalances = more costs |
| `top_n` | 2–4 coins | Integer; 3 means 3 long + 3 short out of 10 coins (30% each leg) |

In [ ]:
# ── search ranges ─────────────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
#
# pct_long / pct_short = fraction of the eligible universe to put in each leg.
# Inside the strategy: n_long  = max(1, int(pct_long  * universe_size))   (floor)
#                     n_short = max(1, int(pct_short * universe_size))   (floor)
# Decile portfolios (10%) are the academic standard for XS momentum;
# 20% is a common variant. Beyond ~30% the ranking signal degrades fast.
PARAM_DEFS = {
    'J':         ('int',   7,    45),    # formation / lookback window (days)
    'K':         ('int',   7,    21),    # holding period (days)
    'pct_long':  ('float', 0.05, 0.3),  # fraction of universe in long leg
    'pct_short': ('float', 0.05, 0.3),  # fraction of universe in short leg
}

# ── fixed params ──────────────────────────────────────────────────────────────
# Start empty — anchor stable parameters (CV < 0.15) after the first stability run.
FIXED_PARAMS = {}

# Example after a stability run:
# FIXED_PARAMS = {
#     'pct_long':  0.20,
#     'pct_short': 0.20,
# }

---
## Strategy

**Ranking signal — rolling Sharpe of raw returns (xs_1 family):**
- Daily returns: `r[t] = r_coin[t]`
- Signal: J-bar Sharpe `μ(r) / σ(r)`, evaluated at bar `t-1` (no lookahead)
- *No BTC factor stripping* — that's the xs_3 family. Differs from xs_3_i_r
  only in this signal block.

**Selection (unchanged):**
- Long top `pct_long × universe` by composite, short bottom `pct_short × universe`.
- `n_long` and `n_short` derived per-rebalance from current universe size.

**Position weighting (NEW — inverse-vol):**
- For each coin in the long leg, weight = `(1/σ_coin) / Σ(1/σ_other_longs)`.
- Same for short leg, computed independently.
- `σ_coin` = rolling stdev of daily perp returns over `IV_VOL_WINDOW` bars,
  evaluated at the formation date (`r-1`) — no lookahead.
- Leg weights still sum to 1 → still 50/50 at the leg level → still dollar-neutral
  in expectation. Only the *intra-leg* allocation changes.

**Why this composes well with the rolling-Sharpe signal:** the signal already
penalises high-vol coins via its denominator. Inverse-vol weighting extends
that penalty to the *position size* — coins that pass the ranking with high
total vol get smaller weight. Two complementary layers of vol-awareness.

**Edge cases handled:**
- Coin with NaN vol (insufficient history) → weight 0 in that leg.
- All NaN vols in a leg (shouldn't happen given universe filter, but defended) →
  fall back to equal weight.
- Coin with NaN return mid-holding-period → weight excluded from denominator at
  that bar; remaining basket fills in pro-rata.

**Cost / turnover model:** unchanged. Engine applies `trade_cost = turnover × COST`.


## Regime-tilted asymmetric leg sizing (Layer 1 — continuous intensity)

On top of the inverse-vol composite signal, leg balance is **asymmetric AND
continuously scaled** based on BTC regime:

**Continuous tilt formula:**
- *direction*  = `+1` if `BTC > EMA-19` else `−1`
- *intensity*  = `min(ADX-14 / 50, 1.0)` ∈ [0, 1]
- *tilt*       = direction × intensity × `MAX_*_TILT` (asymmetric)

| BTC direction + ADX | long_weight | short_weight | Net tilt |
|---|---|---|---|
| Bull, ADX = 50+ (saturated) | 0.55  | 0.45  | +0.05  (max bull) |
| Bull, ADX ≈ 25 (median)     | 0.525 | 0.475 | +0.025 (half-bull) |
| Bull, ADX ≈ 10 (chop)       | 0.51  | 0.49  | +0.01  (mild bull) |
| Bear, ADX ≈ 10 (chop)       | 0.49  | 0.51  | −0.01  (mild bear) |
| Bear, ADX ≈ 25 (median)     | 0.475 | 0.525 | −0.05  (half-bear) |
| Bear, ADX = 50+ (saturated) | 0.40  | 0.60  | −0.10  (max bear) |

Bear tilt at saturation is twice the bull tilt by design — this strategy's
role is hedging long-only momentum books, not adding bull-market beta.
Per-bar returns become:

```
strategy_returns[t] = long_weight[t] · long_basket[t] − short_weight[t] · short_basket[t]
```

Inner weighting (inverse vol within each leg) and outer weighting (regime-tilted
leg balance) are independent — they compose cleanly. Inner weights determine HOW
each leg is composed; outer weights determine HOW the two legs are combined.

**Tilt magnitudes are fixed constants** (cell-3), NOT in `PARAM_DEFS`. Optuna
would otherwise pick the most aggressive tilt that maximises in-sample Sharpe —
turning a hedge into a directional bet disguised as a hedge.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Strategy — built from infra factory (infrastructure/walkforward/xs_strategy.py)
# ══════════════════════════════════════════════════════════════════════════════
my_strategy = make_xs_strategy(
    panel             = panel,
    volume            = volume,
    meta              = meta,
    uf_top_n          = UF_TOP_N,
    uf_min_volume     = UF_MIN_VOLUME,
    uf_min_age        = UF_MIN_AGE_DAYS,
    uf_volume_window  = UF_VOLUME_WINDOW,
    iv_vol_window     = IV_VOL_WINDOW,
    regime_tilt_panel = regime_tilt_panel,
    signal_kind       = 'rolling_sharpe',
)


---
## Run Walk-Forward

**Fold setup for daily data:**

| Parameter | Value | Rationale |
|---|---|---|
| `TRAIN_BARS` | 1050 | ~3 years; many rebalances per fold at K=7-21 |
| `TEST_BARS` | 285 | ~9.5 months; ~3.7:1 train:test ratio |
| `BURNIN_BARS` | 50 | **MUST be >= max(J) + 1** for the rolling-Sharpe signal (single rolling). With max J = 45, 46 is the minimum; 50 leaves a small safety margin. Matches xs_momentum_wf / xs_1 / xs_2. |
| `N_TRIALS` | 300 | 4 free params (J, K, pct_long, pct_short) |
| `cost` | **COST** | Per-leg fraction applied via `turnover × COST` in the engine |

**Why BURNIN_BARS = 50 here:** the rolling-Sharpe signal is a SINGLE rolling(J) operation, so `mom_signal` is NaN only until bar `J+1`. With max J = 45 in PARAM_DEFS, BURNIN_BARS = 50 fully covers the warmup with a small safety margin. No double-rolling overflow concern.

**Cost flow:** `strategy_fn` returns gross `strategy_returns` plus a `turnover`
column. The engine computes `trade_cost = turnover × cost` at each rebalance bar.
Costs propagate correctly through Optuna trials, OOS evaluation, plateau/perturbation
analysis, and `cost_stress_test()` without any special handling.

**Custom `reject_fn`:** `num_trades = 0` for this strategy (position=1 everywhere,
so no position changes are detected). The default filter `num_trades < 7` would
reject all trials. Sharpe ≥ 0 and drawdown are used as quality gates instead.

In [ ]:
# ── walk-forward windows ──────────────────────────────────────────────────────
TRAIN_BARS  = 1050   # ~3 years daily
TEST_BARS   = 285    # ~9.5 months
BURNIN_BARS = 50     # xs_1 family uses a SINGLE rolling(J) operation (rolling
                     # Sharpe of raw returns), so warmup is J + 1.  With max
                     # J = 45 in PARAM_DEFS, 46 is the minimum; 50 leaves a
                     # small safety margin.  Same burnin as xs_momentum_wf /
                     # xs_1 / xs_2 — consistent across single-rolling families.
N_TRIALS    = 300

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
def score_fn(m):
    SHARPE_MAX = 3.0
    CALMAR_MAX = 6.0
    RETURN_MAX = 10.0

    calmar = (m['total_return'] / abs(m['max_drawdown'])
              if m['max_drawdown'] != 0 else 0.0)

    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)

    return 0.60 * s + 0.25 * c + 0.15 * r


# ── REJECTION CRITERIA ────────────────────────────────────────────────────────
# Do NOT check num_trades — always 0 for this strategy (position=1 everywhere).

def reject_fn(m):
    if m is None:                   return True
    if m['sharpe_ratio'] < 0.0:     return True
    if m['max_drawdown'] < -0.80:   return True
    if m['total_return'] < 0.0:     return True
    return False


results = walk_forward(
    df           = driver_df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    train_bars   = TRAIN_BARS,
    test_bars    = TEST_BARS,
    burnin_bars  = BURNIN_BARS,
    n_trials     = N_TRIALS,
    cost         = COST,       # applied by engine as: trade_cost = turnover * COST
    score_fn     = score_fn,
    reject_fn    = reject_fn,
    save_csv     = None,
)


---
## Granular Results and Parameter Stability

Per-fold IS vs OOS performance. Compare `train_*` vs `test_*` to assess overfitting.

| Column | Description |
|---|---|
| `*_sharpe` `*_return` `*_drawdown` `*_calmar` | Core performance metrics |
| `optuna_score` | Best score achieved on training window |
| `param_J` `param_K` `param_pct_long` `param_pct_short` | Best parameter values per fold |

**Note:** `test_trades`, `test_winrate`, `test_profit_factor` will all be 0 — the
engine logs no trades because `position = 1` everywhere (see Cell 4). Use Sharpe,
return, drawdown, **and the long/short attribution table further below** as the
primary OOS quality signals for this strategy.

**Consensus Parameters** — anchor parameters with CV < 0.15 in `FIXED_PARAMS`.
For percentile-based XS momentum, `pct_long` and `pct_short` are often the most
stable since the strategy is now scale-invariant to universe size.

In [ ]:
# ── fold summary table ────────────────────────────────────────────────────────
display_cols = [
    'train_start', 'train_end',
    'test_start',  'test_end',
    'train_sharpe', 'test_sharpe',
    'train_return', 'test_return',
    'train_drawdown', 'test_drawdown',
    'optuna_score',
    'param_J', 'param_K', 'param_pct_long', 'param_pct_short',
]
display(results['results_df'][display_cols].round(3))

# ── parameter stability ───────────────────────────────────────────────────────
stability = results['stability_df'].copy()
stability['stable'] = stability['stable'].map({True: 'yes', False: ''})
stability['fixed']  = stability['fixed'].map({True: 'yes', False: ''})
stability = stability[['param', 'median', 'std', 'cv', 'stable', 'fixed']].round(3)
print('\nParameter stability (CV < 0.15 = stable candidate for fixing):')
display(stability.sort_values('cv'))

# ── consensus params ──────────────────────────────────────────────────────────
stable = results['stability_df'][results['stability_df']['cv'] < 0.15]

print('Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:')
for _, row in stable.iterrows():
    v = results['consensus_params'][row['param']]
    v_fmt = (int(round(v)) if isinstance(v, float) and v == int(v)
             else round(v, 4) if isinstance(v, float) else v)
    print(f"    '{row['param']}': {v_fmt},")

print('\nConsensus parameters (median across folds):')
for k, v in results['consensus_params'].items():
    print(f'  {k:<12} = {round(v, 4) if isinstance(v, float) else v}')

---
## Parameter Robustness Checks

### Plateau Analysis
Sweeps each free parameter across its full range (holding others at consensus),
evaluating the score at each point over the full date range.

| Verdict | Plateau % | Meaning |
|---|---|---|
| Robust | ≥ 60% | Score stays near peak across most of the range |
| Moderate | 30–60% | Some sensitivity — watch if it degrades further after fixing |
| FRAGILE | < 30% | Strategy depends critically on hitting the exact value — consider anchoring |
| N/A | — | Every sweep point failed reject_fn — strategy completely intolerant on this axis |

### Perturbation Test
Randomly jitters **all** free parameters simultaneously by ±5/10/20% of their range.
Tests whether the optimum is a broad hill or a narrow spike in the joint parameter space.

> With only 3 integer parameters, this test is less informative than for 15-param
> strategies. Focus on the plateau % instead.

In [ ]:
# ── 1-D sensitivity sweeps around consensus params ─────────────────────────
sweep_results = plateau_analysis(
    df           = driver_df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = 0.0,         # costs baked into strategy_returns
    reject_fn    = reject_fn,   # required: default reject_fn checks num_trades < 7 which is always 0
    n_steps      = 20,
)

# ── text verdicts ──────────────────────────────────────────────────────────
verdict_df = plateau_summary(
    sweep_results,
    base_params  = results['consensus_params'],
    stability_df = results['stability_df'],
    threshold    = 0.20,
)

# ── neighbourhood perturbation ─────────────────────────────────────────────
# With only 3 integer parameters the perturbation may hit boundary values often.
# Treat degradation > 20% at ±10% offset as a fragility signal.
perturb_df = perturbation_test(
    df           = driver_df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = 0.0,
    reject_fn    = reject_fn,   # required: default reject_fn checks num_trades < 7 which is always 0
    pct_offsets  = (0.05, 0.10, 0.20),
    n_samples    = 50,
)

In [ ]:

# ── plateau sweep charts ──────────────────────────────────────────────────────
plot_plateau_analysis(
    sweep_results    = sweep_results,
    consensus_params = results['consensus_params'],
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    threshold        = 0.20,
    show             = False,
    save_html        = None,
)


---
## Results Charts and Cost Stress Test

**OOS equity chart** — combined OOS period stitched from all folds, overlaid with the
equal-weight universe basket (benchmark).

**Why the equal-weight basket is the right benchmark for this strategy:**  
A dollar-neutral XS momentum portfolio (long top-N, short bottom-N, equal weight) has
**zero net market exposure** by construction — the long and short legs cancel in aggregate.
This means BTC buy-and-hold is *not* a valid comparison: if the whole market rises uniformly,
a dollar-neutral strategy should earn approximately zero from that move.

The equal-weight basket represents the passive return from holding all universe coins equally,
with no selection or ranking. Any outperformance over this baseline is *pure cross-sectional
alpha* — return that comes solely from the ranking signal correctly identifying relative
winners and losers, not from riding overall market direction.

Reading the chart:
- **Benchmark rises, strategy lags** → the short leg is the drag; you are short coins that
  are rising, just slower than the longs. The strategy is still doing its job, but market
  beta inside the long leg does not offset the short-leg drag.
- **Benchmark falls, strategy holds or rises** → the short leg is working; the coins you
  are short are falling harder than the longs, generating positive spread return.
- **Both fall together** → a broad market crash overwhelms cross-sectional differentiation;
  expected for XS momentum in high-correlation bear regimes where all assets move together.

**Cost stress test** — re-runs the combined OOS backtest at escalating cost multipliers.
Works correctly because `oos_combined_df` carries the `turnover` column: the engine applies
`trade_cost = turnover × cost` at each rebalance bar, so raising `cost` correctly scales
the burden on high-turnover periods without re-running the strategy.

In [ ]:
# ── OOS equity + fold charts ──────────────────────────────────────────────────
plot_walk_forward_results(
    results         = results,
    param_defs      = PARAM_DEFS,
    fixed_params    = FIXED_PARAMS,
    benchmark_data  = ew_df,       # equal-weight universe basket
    show            = True,
    save_html_dir   = None,
    show_fold_perf  = False,
    show_param_evol = False,
    show_oos_equity = True,
    show_trades     = False,
)

# ── transaction cost stress test ──────────────────────────────────────────────
if results['oos_combined_df'] is not None:
    cost_df = cost_stress_test(
        oos_combined_df  = results['oos_combined_df'],
        cost_multipliers = (1.0, 1.5, 2.0, 3.0),
        base_cost        = COST,
    )
else:
    print('No combined OOS dataframe — skip cost stress test.')

---
## Long/Short Attribution Diagnostics

Cross-sectional-specific diagnostics that replace trade-centric metrics
(meaningless here since `position=1` always — the engine logs zero trades).

**Attribution table** — Sharpe and total return decomposed into:
- `long_sharpe` / `long_total_return` — performance of being 100% long the long basket
- `short_sharpe` / `short_basket_total` — performance of being 100% short the short basket
- `spread_sharpe` / `spread_total` — dollar-neutral spread (long − short, 1× leverage)
- `net_sharpe` — combined post-cost return (matches the OOS Sharpe in the fold table)
- `hit_rate` — fraction of rebalance periods where (long − short) > 0
- `mean_turnover` — average fractional turnover at rebalance bars (range 0–2)
- `avg/min/max_universe_size` — # coins available at rebalance over the OOS period

**Single most important read:** if `short_sharpe < 0` consistently across runs, the
short leg is a drag rather than a profit centre. That argues for dropping it
(long-only) or re-thinking the ranking (the bottom of the universe might not be
mean-reverting losers — it might be coins that already crashed and bounce).

**4-panel chart** —
1. Cumulative equity by leg (long, −short, combined net) — visual long/short attribution
2. Cumulative spread (long − short) — the raw momentum premium harvested
3. Turnover per rebalance — high turnover = unstable rankings = noisy signal
4. Universe size over time — flags degraded regimes where ranking is meaningless

In [ ]:
# ── long/short attribution table + diagnostic chart ──────────────────────────
oos = results['oos_combined_df']

if oos is None or 'long_ret' not in oos.columns:
    print('No OOS dataframe with attribution columns — skip diagnostics.')
else:
    attr = compute_attribution(oos, ann=365)

    print('═' * 60)
    print('LONG/SHORT ATTRIBUTION')
    print('═' * 60)
    for k, v in attr.items():
        if isinstance(v, float):
            if 'sharpe' in k:
                print(f'  {k:<22} {v:>8.3f}')
            elif 'return' in k or 'total' in k or 'rate' in k or 'turnover' in k:
                print(f'  {k:<22} {v:>8.2%}' if abs(v) < 100 else f'  {k:<22} {v:>8.3f}')
            else:
                print(f'  {k:<22} {v:>8.2f}')
        else:
            print(f'  {k:<22} {v:>8}')

    plot_attribution(oos,
                     benchmark_data=ew_df,   # equal-weight basket → strips beta from leg returns
                     show=True, save_html=None,
                     title='XS Momentum — Long/Short Diagnostics (OOS)')

---
## Bear-Hedge Diagnostic — Conditional Regime Performance

Tests whether this XS strategy genuinely works as a hedge to long-only /
momentum strategies during BTC drawdowns. Splits OOS bars by sign of BTC's
daily return and computes per-regime alphas + strategy Sharpe.

**What to read:**
- `short_α(down)` > `short_α(up)` → short leg generates real return when BTC
  is falling. **This is the bear-hedge condition.**
- `strat_Sharpe(down)` > 0 → strategy makes money in BTC-down regimes. If
  this is positive while your other momentum strategies struggle in those
  same windows, XS earns its slot in the portfolio.
- `|corr(strategy, BTC)|` near 0 → dollar-neutral construction is holding,
  no hidden BTC beta leaking through.

**If short_α(down) ≤ short_α(up):** the short leg is *not* a hedge — it's just
correlated alpha. Either redesign the short signal or accept the strategy as
a separate alpha source rather than a hedge. **Don't asymmetrically size
toward the short leg until this test confirms it works in BTC-down.**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Bear-hedge diagnostic — performance conditional on BTC daily-return sign
# (infrastructure/walkforward/ls_diagnostics.py)
# ══════════════════════════════════════════════════════════════════════════════
oos = results['oos_combined_df']
bh = bear_hedge_diagnostic(oos, panel['BTC'], ann=365)


---
## Regime Quadrant Diagnostic — BTC Trend × Trend Strength

Splits the OOS period into four regime quadrants and reports strategy performance
in each. Reveals where the strategy makes its money vs where it bleeds — a
prerequisite to designing regime-conditional filters.

**Two regime axes:**

| | Bull *(BTC > 200d MA)* | Bear *(BTC < 200d MA)* |
|---|---|---|
| **Trending** *(ER ≥ median)* | Bull / trending | Bear / trending |
| **Chop**  *(ER < median)*    | Bull / chop      | Bear / chop      |

**Trend-strength = Kaufman Efficiency Ratio (14-day)** on BTC close:
`ER = |close[t] − close[t−14]| / Σ|close[i] − close[i−1]|`. Range [0, 1]; high
= clean directional move, low = same net move achieved through choppy back-and-
forth. Conceptually equivalent to ADX (which also measures trend persistence)
but computable from Close alone — the cache doesn't store H/L, so true ADX
would require a cache rebuild. ER captures ~95% of the same regime signal.

**What to read:**
- **Returns positive in 3 quadrants, negative in 1** → high-value regime filter
  candidate (turn strategy off in the bad quadrant).
- **Returns negative in ≥2 quadrants** → strategy is fragile across regimes;
  filter alone won't fix it, signal needs work.
- **Bear-hedge use case specifically:** want positive Sharpe in *both* bear
  quadrants. Bear/trending only → strategy is a bear-trend hedge (works in
  waterfall declines, fails in slow grinders). Bear/chop only → strategy is a
  bear-mean-reversion hedge (works in choppy bears, fails when BTC capitulates).
- **Bull/trending dominant return** → the strategy is essentially a long-only
  momentum bet riding bull trends. Not a hedge; a directional alpha source.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Regime quadrant diagnostic — BTC trend × trend-strength (Efficiency Ratio)
# (infrastructure/walkforward/ls_diagnostics.py)
# ══════════════════════════════════════════════════════════════════════════════
oos = results['oos_combined_df']
rq = regime_quadrant_diagnostic(oos, panel['BTC'],
                                ma_window=200, er_window=14, ann=365)


---
## Live Execution Notes (forward-looking — for when this goes live)

**Venue:** Binance (single account, no multi-venue complexity).

**Recommended structure:** spot-long + perp-short under cross-margin.
- **Long leg** → Binance spot (just hold the coin)
- **Short leg** → Binance USDT-M perp (open short position)
- Cross-margin lets the spot holdings collateralize the perp shorts → capital efficiency stays close to perp/perp.

**Why this beats perp/perp:**
On XS momentum specifically, the long leg picks the *highest-momentum* coins, which systematically have the *highest* funding rates (long-biased flow). The short leg picks the *lowest-momentum* coins, which usually have low / negative funding rates. So perp/perp pays a lot on longs and only partially offsets via shorts. Spot-long avoids the long-side cost entirely; perp-short still receives any short-side funding. Net funding moves from "drag" to "small positive."

**Funding-rate cache deferred.** With this structure funding is no longer a first-order cost — modeling it precisely is a refinement, not a gate. Build it later only if you also want to backtest a perp/perp variant for comparison. Architecture sketch (when needed):
- `infrastructure/data/funding_cache.py` — mirror `universe_cache.py` (incremental fetch, parquet output)
- `infrastructure/perps/funding.py` — `apply_funding(positions, funding_panel)` helper
- Sign convention: `funding_pnl = -position × funding_rate` (long pays when rate > 0)

**Backtest data note:** the perp cache built by `universe_cache.py` is fine as a price proxy for spot in the backtest. For top-30 liquid Binance coins, spot ↔ perp basis runs ~5–15 bps intraday, well under the noise floor of a daily strategy. Don't build a parallel spot cache just for backtesting purposes.

**Bigger missing items to model before going live (in priority order):**
1. **Slippage on the bottom of the universe.** Rank 25–30 by volume can be thinly traded — assume 5–15 bps per leg vs the close.
2. **Borrow availability** if you ever go spot-margin-short instead of perp-short (some coins are hard-to-borrow; rates spike in stressed regimes).
3. **Funding** — only after the above two are calibrated. For spot-long + perp-short it's a small positive contribution, not a drag.

In [ ]:
# ── save OOS dataframe ────────────────────────────────────────────────────────
# Load in a portfolio notebook with:
#   xs_oos = pd.read_pickle(os.path.join('oos', 'xs_1_i_r_oos.pkl'))
if results['oos_combined_df'] is not None:
    oos_dir  = os.path.join(os.path.dirname(os.path.abspath('xs_momentum_wf.ipynb')), 'oos')
    os.makedirs(oos_dir, exist_ok=True)
    oos_path = os.path.join(oos_dir, 'xs_1_i_r_oos.pkl')
    results['oos_combined_df'].to_pickle(oos_path)
    print(f'Saved OOS dataframe  ({len(results["oos_combined_df"])} bars)  →  {oos_path}')
else:
    print('No combined OOS dataframe to save.')